Pada penelitian ini akan dilakukan eksperimen menggunakan model **Artificial Neural Network (ANN), Support Vector Machine (SVM), dan LightGBM** dengan tiga pendekatan word embedding, yaitu **Word2Vec, FastText, dan GloVe**. Oleh karena itu, diperlukan proses pre-processing teks yang ekstensif untuk meningkatkan kualitas data sebelum digunakan dalam proses pembelajaran model.

Pada tahap pre-processing, penelitian ini memanfaatkan beberapa sumber daya NLP Bahasa Indonesia yang tersedia pada repositori **[NLP Bahasa Resources yang dikembangkan oleh Louis Owen](https://github.com/louisowen6/NLP_bahasa_resources)**. 
* Untuk proses **normalisasi kata tidak baku** (***slang normalization***), dibuat sebuah modul Python bernama `slang_words`. Modul ini menggabungkan daftar kata slang yang berasal dari file `combined_root_words.txt` pada repositori tersebut dengan kata-kata slang tambahan yang ditemukan pada dataset penelitian ini. Proses ini bertujuan untuk meningkatkan cakupan normalisasi kata tidak baku yang muncul pada data ulasan. 
* Untuk proses **penghapusan stopwords**, digunakan daftar stopwords dari library **Sastrawi** yang kemudian dikombinasikan dengan daftar stopwords dari file `combined_stop_words.txt` pada repositori tersebut, serta stopwords tambahan yang ditemukan secara empiris pada dataset.
* Daftar kata dasar pada file `combined_root_words.txt` digunakan sebagai referensi untuk mendeteksi apakah suatu kata dalam dataset masih merupakan bentuk tidak baku atau belum dinormalisasi. Proses ini dilakukan secara iteratif hingga sebagian besar kata dalam dataset telah dinormalisasi ke bentuk baku.

# Import Dependencies

Sebelum melakukan proses pre-processing, beberapa library Python perlu diinstal, yaitu:
* **emoji**<br>
Digunakan untuk mendeteksi dan memisahkan emoji dalam teks sehingga emoji dapat diperlakukan sebagai token terpisah dalam proses tokenisasi.
* **Sastrawi**<br>
ibrary NLP Bahasa Indonesia yang digunakan untuk proses **stemming** dan **stopword removal**.

In [ ]:
!pip install emoji Sastrawi

In [ ]:
import pickle
import string

import emoji
import nltk
import pandas as pd
from nltk import word_tokenize
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from tqdm import tqdm

# custom modules
import slang_words

# Mengunduh model tokenizer NLTK yang diperlukan oleh fungsi word_tokenize
nltk.download("punkt") 
nltk.download("punkt_tab")

# Aktifkan integrasi tqdm dengan pandas
tqdm.pandas()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
def separate_emoji(text):
  """
  Separate emojis in a text by surrounding them with spaces.

  This function uses the `emoji.replace_emoji()` utility to detect emojis
  in a string and add spaces before and after each emoji. This is useful
  for text preprocessing tasks (e.g., NLP tokenization) so that emojis
  are treated as separate tokens instead of being attached to words.

  Parameters
  ----------
  text : str
    Input text that may contain emojis.

  Returns
  -------
  str
    The processed text where each emoji is separated by spaces.

  Raises
  ------
  ImportError
    If the `emoji` package is not installed.

  Notes
  -----
  This function requires the external library `emoji`. Install it with:

    pip install emoji

  Examples
  --------
  >>> separate_emoji("I love pizza🍕!")
  'I love pizza 🍕 !'
  """
  return emoji.replace_emoji(text, replace=lambda e, data: f" {e} ")

# Buka file combined_root_words.txt dan baca isinya
with open("combined_root_words.txt", "r", encoding="utf-8") as file:
  # Membaca setiap baris, membersihkan spasi/enter (\n) di awal/akhir kata,
  listKataDasar = [line.strip() for line in file if line.strip()]

listKataDasar = set(listKataDasar)

# Buka file combined_stop_words.txt dan baca isinya
with open("combined_stop_words.txt", "r", encoding="utf-8") as file:
  # Membaca setiap baris, membersihkan spasi/enter (\n) di awal/akhir kata,
  listOfStopwords = [line.strip() for line in file if line.strip()]

listOfStopwords = set(listOfStopwords)

def get_stemmed_term(term: str):
  """
  Retrieve the stemmed form of a term using a cache dictionary.

  This function stems a given term using a stemmer and stores the result
  in a cache dictionary to avoid repeated stemming of the same term.
  If the term has already been processed, the cached result is returned
  directly, improving performance during large-scale text preprocessing.

  Parameters
  ----------
  term : str
    The word or token to be stemmed.

  Returns
  -------
  str
    The stemmed version of the input term.

  Notes
  -----
  This function assumes that two global variables exist:
  - `stemmer` : an object implementing a `.stem()` method.
  - `stem_cache` : a dictionary used to store previously stemmed terms.

  Examples
  --------
  >>> get_stemmed_term("berlari")
  'lari'

  >>> get_stemmed_term("memakan")  # retrieved from cache on repeated calls
  'makan'
  """
  if term not in stem_cache:
    stem_cache[term] = stemmer.stem(term)
  return stem_cache[term]

# Load Data

In [ ]:
# Membaca dataset hasil scraping/review yang sudah diberi label sentimen
df = pd.read_csv('labelled_review_bsi_mobile.csv')

# Mengambil kolom yang relevan untuk analisis
df = df[['reviewId', 'content', 'sentiment']]

df.sample(10, random_state=2)

,reviewId,content,sentiment
9305,75502cae-c7b1-4e91-a5f1-6699212e699b,Saya nga bisa aktivitasi kenapa ya?,negative
10889,c2cbda72-e24f-456c-a316-9569a75fb307,"Min, m banking ku udah 3 hari ga bisa dipake t...",negative
7593,34b4b30d-70df-4122-b021-4bdcc5356556,"Dari segi aplikasi yg baru, sangat tidak nyama...",negative
2330,4c7fe7fb-ace4-4088-8c62-13b6f7c44240,Aplikasinya baik sekali,positive
5356,e77d8f3e-36ee-4d9b-8a1e-dcb75730d15a,"Sangat membantu, minim gangguan dan juga sanga...",positive
8299,d8351a9b-18e7-4bbf-bf1b-a9a74bfe9494,"Sejak update ke BSI fersi terbaru, banyak seka...",negative
10570,ad384e26-785e-4df3-a6c5-50b6086f4361,Aplikasinya jelek. Lola dan banyak fitur yang ...,negative
7434,7c46d465-d281-42c6-a6cf-ce12032ddaa6,Knp apk ini gk bisa di update,negative
9685,9db1741d-a8c4-42e9-8f52-9d4620792d38,ok,negative
5278,1e1f249a-4609-4267-b2df-d05d5786bcf0,"Keren nih aplikasinya, mempermudah transaksi😍",positive


In [ ]:
# Melihat informasi dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14295 entries, 0 to 14294
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   reviewId   14295 non-null  object
 1   content    14295 non-null  object
 2   sentiment  14295 non-null  object
dtypes: object(3)
memory usage: 335.2+ KB


Dataset yang digunakan terdiri dari **14.295 ulasan** pengguna yang diperoleh dari Google Play Store dan telah dilabeli sentimen. Dataset ini memiliki tiga kolom utama yaitu:
* `reviewId` sebagai identitas ulasan
* `content` yang berisi teks ulasan pengguna
* `sentiment` yang berisi label sentimen

# Text Pre-processing

## Case Folding

Karena penelitian ini menggunakan pendekatan **machine learning konvensional** (bukan model berbasis transformer seperti BERT), maka dilakukan proses **case folding**, yaitu mengubah seluruh teks menjadi huruf kecil.

Proses ini bertujuan untuk menghindari duplikasi fitur yang disebabkan oleh perbedaan huruf kapital, misalnya kata "Bagus" dan "bagus" yang sebenarnya memiliki makna yang sama.

In [ ]:
# Mengubah seluruh teks menjadi huruf kecil (case folding)
# agar kata yang sama tidak dianggap berbeda
df['content'] = df['content'].str.lower()

df.sample(10, random_state=2)

,reviewId,content,sentiment
9305,75502cae-c7b1-4e91-a5f1-6699212e699b,saya nga bisa aktivitasi kenapa ya?,negative
10889,c2cbda72-e24f-456c-a316-9569a75fb307,"min, m banking ku udah 3 hari ga bisa dipake t...",negative
7593,34b4b30d-70df-4122-b021-4bdcc5356556,"dari segi aplikasi yg baru, sangat tidak nyama...",negative
2330,4c7fe7fb-ace4-4088-8c62-13b6f7c44240,aplikasinya baik sekali,positive
5356,e77d8f3e-36ee-4d9b-8a1e-dcb75730d15a,"sangat membantu, minim gangguan dan juga sanga...",positive
8299,d8351a9b-18e7-4bbf-bf1b-a9a74bfe9494,"sejak update ke bsi fersi terbaru, banyak seka...",negative
10570,ad384e26-785e-4df3-a6c5-50b6086f4361,aplikasinya jelek. lola dan banyak fitur yang ...,negative
7434,7c46d465-d281-42c6-a6cf-ce12032ddaa6,knp apk ini gk bisa di update,negative
9685,9db1741d-a8c4-42e9-8f52-9d4620792d38,ok,negative
5278,1e1f249a-4609-4267-b2df-d05d5786bcf0,"keren nih aplikasinya, mempermudah transaksi😍",positive


## Emoji Handling

Dalam analisis sentimen, **emoji sering kali merepresentasikan ekspresi emosi pengguna**, sehingga dapat memberikan informasi tambahan terhadap polaritas sentimen suatu ulasan. Oleh karena itu, emoji pada penelitian ini tidak dihapus, melainkan dipisahkan dari teks agar dapat diperlakukan sebagai token tersendiri pada proses tokenisasi.

In [ ]:
# Memisahkan emoji dari teks agar dapat diproses sebagai token terpisah
df["content"] = df["content"].apply(separate_emoji)

df.sample(10, random_state=2)

,reviewId,content,sentiment
9305,75502cae-c7b1-4e91-a5f1-6699212e699b,saya nga bisa aktivitasi kenapa ya?,negative
10889,c2cbda72-e24f-456c-a316-9569a75fb307,"min, m banking ku udah 3 hari ga bisa dipake t...",negative
7593,34b4b30d-70df-4122-b021-4bdcc5356556,"dari segi aplikasi yg baru, sangat tidak nyama...",negative
2330,4c7fe7fb-ace4-4088-8c62-13b6f7c44240,aplikasinya baik sekali,positive
5356,e77d8f3e-36ee-4d9b-8a1e-dcb75730d15a,"sangat membantu, minim gangguan dan juga sanga...",positive
8299,d8351a9b-18e7-4bbf-bf1b-a9a74bfe9494,"sejak update ke bsi fersi terbaru, banyak seka...",negative
10570,ad384e26-785e-4df3-a6c5-50b6086f4361,aplikasinya jelek. lola dan banyak fitur yang ...,negative
7434,7c46d465-d281-42c6-a6cf-ce12032ddaa6,knp apk ini gk bisa di update,negative
9685,9db1741d-a8c4-42e9-8f52-9d4620792d38,ok,negative
5278,1e1f249a-4609-4267-b2df-d05d5786bcf0,"keren nih aplikasinya, mempermudah transaksi 😍",positive


## Special Characters Removal

Pada tahap ini dilakukan pembersihan teks dengan menghapus **angka, tanda baca, dan spasi berlebih**. Elemen-elemen tersebut dianggap tidak memberikan kontribusi signifikan terhadap proses klasifikasi sentimen pada pendekatan yang digunakan dalam penelitian ini.

In [ ]:
df["content"] = (
    df["content"]
    .str.replace(r"\d+", "", regex=True) # Menghapus angka
    .str.replace(f"[{string.punctuation}]", "", regex=True) # Menghapus tanda baca
    .str.replace(r"\s+", " ", regex=True) # Mengganti multiple whitespace menjadi satu spasi
    .str.strip()
)

df.sample(10, random_state=2)

,reviewId,content,sentiment
9305,75502cae-c7b1-4e91-a5f1-6699212e699b,saya nga bisa aktivitasi kenapa ya,negative
10889,c2cbda72-e24f-456c-a316-9569a75fb307,min m banking ku udah hari ga bisa dipake tran...,negative
7593,34b4b30d-70df-4122-b021-4bdcc5356556,dari segi aplikasi yg baru sangat tidak nyaman...,negative
2330,4c7fe7fb-ace4-4088-8c62-13b6f7c44240,aplikasinya baik sekali,positive
5356,e77d8f3e-36ee-4d9b-8a1e-dcb75730d15a,sangat membantu minim gangguan dan juga sangat...,positive
8299,d8351a9b-18e7-4bbf-bf1b-a9a74bfe9494,sejak update ke bsi fersi terbaru banyak sekal...,negative
10570,ad384e26-785e-4df3-a6c5-50b6086f4361,aplikasinya jelek lola dan banyak fitur yang m...,negative
7434,7c46d465-d281-42c6-a6cf-ce12032ddaa6,knp apk ini gk bisa di update,negative
9685,9db1741d-a8c4-42e9-8f52-9d4620792d38,ok,negative
5278,1e1f249a-4609-4267-b2df-d05d5786bcf0,keren nih aplikasinya mempermudah transaksi 😍,positive


## Normalization

Karena dataset berasal dari ulasan pengguna di Google Play Store, banyak ditemukan **kata tidak baku, singkatan, maupun bahasa slang**. Variasi penulisan tersebut dapat menyebabkan jumlah fitur meningkat secara signifikan dan mengurangi konsistensi representasi kata. Oleh karena itu, dilakukan proses **normalisasi kata slang** untuk mengubah kata tidak baku menjadi bentuk baku.

In [ ]:
# Mengubah kata slang / tidak baku menjadi bentuk baku
# contoh: "gk" -> "tidak"
df["content"] = df["content"].apply(slang_words.fix_slangwords)

df.sample(10, random_state=2)

,reviewId,content,sentiment
9305,75502cae-c7b1-4e91-a5f1-6699212e699b,saya nga bisa aktivitasi kenapa iya,negative
10889,c2cbda72-e24f-456c-a316-9569a75fb307,min m banking ku sudah hari tidak bisa dipake ...,negative
7593,34b4b30d-70df-4122-b021-4bdcc5356556,dari segi aplikasi yang baru sangat tidak nyam...,negative
2330,4c7fe7fb-ace4-4088-8c62-13b6f7c44240,aplikasinya baik sekali,positive
5356,e77d8f3e-36ee-4d9b-8a1e-dcb75730d15a,sangat membantu minim gangguan dan juga sangat...,positive
8299,d8351a9b-18e7-4bbf-bf1b-a9a74bfe9494,sejak update ke bsi fersi terbaru banyak sekal...,negative
10570,ad384e26-785e-4df3-a6c5-50b6086f4361,aplikasinya jelek lambat berfikir dan banyak f...,negative
7434,7c46d465-d281-42c6-a6cf-ce12032ddaa6,kenapa apk ini gk bisa di update,negative
9685,9db1741d-a8c4-42e9-8f52-9d4620792d38,ok,negative
5278,1e1f249a-4609-4267-b2df-d05d5786bcf0,keren ini aplikasinya mempermudah transaksi 😍,positive


## Tokenization

Pada tahap ini dilakukan tokenisasi, yaitu proses memecah teks ulasan menjadi unit-unit kata (token). Tokenisasi diperlukan agar setiap kata dapat diproses lebih lanjut dalam tahap preprocessing berikutnya maupun dalam proses pembentukan fitur.

In [ ]:
# Memecah setiap ulasan menjadi token-token kata
df["content"] = df["content"].apply(word_tokenize)

df.sample(10, random_state=2)

,reviewId,content,sentiment
9305,75502cae-c7b1-4e91-a5f1-6699212e699b,"[saya, nga, bisa, aktivitasi, kenapa, iya]",negative
10889,c2cbda72-e24f-456c-a316-9569a75fb307,"[min, m, banking, ku, sudah, hari, tidak, bisa...",negative
7593,34b4b30d-70df-4122-b021-4bdcc5356556,"[dari, segi, aplikasi, yang, baru, sangat, tid...",negative
2330,4c7fe7fb-ace4-4088-8c62-13b6f7c44240,"[aplikasinya, baik, sekali]",positive
5356,e77d8f3e-36ee-4d9b-8a1e-dcb75730d15a,"[sangat, membantu, minim, gangguan, dan, juga,...",positive
8299,d8351a9b-18e7-4bbf-bf1b-a9a74bfe9494,"[sejak, update, ke, bsi, fersi, terbaru, banya...",negative
10570,ad384e26-785e-4df3-a6c5-50b6086f4361,"[aplikasinya, jelek, lambat, berfikir, dan, ba...",negative
7434,7c46d465-d281-42c6-a6cf-ce12032ddaa6,"[kenapa, apk, ini, gk, bisa, di, update]",negative
9685,9db1741d-a8c4-42e9-8f52-9d4620792d38,[ok],negative
5278,1e1f249a-4609-4267-b2df-d05d5786bcf0,"[keren, ini, aplikasinya, mempermudah, transak...",positive


## Stopwords Removal

Karena penelitian ini menggunakan pendekatan **machine learning konvensional**, maka dilakukan proses **penghapusan stopwords**. Stopwords merupakan kata-kata umum yang sering muncul dalam teks namun biasanya tidak memiliki kontribusi signifikan terhadap makna utama suatu kalimat. Penghapusan stopwords bertujuan untuk mengurangi noise pada data serta menurunkan **dimensionalitas fitur** yang dihasilkan dari proses ekstraksi fitur.

Selain menggunakan daftar stopwords dari repo Louis Owen dan library Sastrawi, penelitian ini juga menambahkan **beberapa stopwords kustom** yang sering muncul pada dataset, seperti kata "bsi", "mobile", dan "aplikasi". Kata-kata tersebut dianggap tidak memberikan kontribusi yang signifikan dalam menentukan polaritas sentimen karena muncul secara umum pada hampir seluruh ulasan.

In [ ]:
# Menambahkan daftar stopword Bahasa Indonesia dari Sastrawi dan
# stopword kustom yang sering muncul pada dataset
stopwords_factory = StopWordRemoverFactory()
listOfStopwords.update(stopwords_factory.get_stop_words())
listOfStopwords.update(["apk", "aplikasi", "aplikasinya", "app", "bahkan", "bank", "banking", "bsi", "byond", "min", "mobile", "nya", "sejak", "sih", "warahmatullahi", "wabarakatuh"])

In [ ]:
# Menghapus kata-kata stopword dari setiap token
df["content"] = df["content"].apply(lambda tokens: [t for t in tokens if t not in listOfStopwords])

df.sample(10, random_state=2)

,reviewId,content,sentiment
9305,75502cae-c7b1-4e91-a5f1-6699212e699b,"[nga, aktivitasi, iya]",negative
10889,c2cbda72-e24f-456c-a316-9569a75fb307,"[m, ku, dipake, transaksi, coba, masuk, log, o...",negative
7593,34b4b30d-70df-4122-b021-4bdcc5356556,"[segi, nyaman, salah, tangkapan, layar, hasil,...",negative
2330,4c7fe7fb-ace4-4088-8c62-13b6f7c44240,[],positive
5356,e77d8f3e-36ee-4d9b-8a1e-dcb75730d15a,"[minim, gangguan, nyaman, terimakasih, pelayan...",positive
8299,d8351a9b-18e7-4bbf-bf1b-a9a74bfe9494,"[update, fersi, terbaru, gangguan, buka, kecewa]",negative
10570,ad384e26-785e-4df3-a6c5-50b6086f4361,"[jelek, lambat, berfikir, fitur, merepotkan]",negative
7434,7c46d465-d281-42c6-a6cf-ce12032ddaa6,"[gk, update]",negative
9685,9db1741d-a8c4-42e9-8f52-9d4620792d38,[],negative
5278,1e1f249a-4609-4267-b2df-d05d5786bcf0,"[keren, mempermudah, transaksi, 😍]",positive


Setelah proses stopword removal, beberapa ulasan dapat menjadi kosong karena seluruh kata pada ulasan tersebut termasuk dalam daftar stopwords. Oleh karena itu, baris data yang tidak memiliki token setelah proses preprocessing dihapus agar tidak mempengaruhi proses pelatihan model.

In [ ]:
# Menghapus baris yang tidak memiliki token setelah proses preprocessing
df = df[df["content"].str.len() > 0]

df.sample(10, random_state=2)

,reviewId,content,sentiment
9026,1b0be493-10ef-4053-9708-bb449f4a902c,"[susah, masuk, frefikasih, wajah, kayal, taikkkk]",negative
11681,fb39dc4b-0e30-4600-b6a8-6dcbd33119a0,[eror],negative
10855,bd3f91b6-254d-4286-8138-5f08809f3445,"[woi, kebuka, ku, uninstal, pas, instal, ulang...",negative
770,74301678-3701-4d6c-aed4-6eca88a67c35,"[hilang, tibaknpadisaat, urgent, digunakan]",negative
3406,9019166f-5afd-45ed-a42b-c4019d286aaf,"[masyaa, allah, puas, prlayanannya]",positive
3262,701974be-4b7d-4758-9e09-3ecea8e31e41,"[simpel, mudah, digunakanbebas, biaya, pengiri...",positive
2562,b76d0167-2e6b-46a9-879e-72f025daf946,"[bagus, mudah, digunakan, mantap]",positive
5955,4a28e683-1d5b-4a1d-b9d7-4e324b48ef50,"[transaksi, jaringan, stabil, gangguanleg, tra...",negative
10174,69b99d7e-6207-4ecb-b718-8e3aef084c1a,"[pengguna, kecewa, sidik, jari, print, dikenal...",negative
9604,4857ddf0-0726-4b0d-8d4f-3176a4ec1e1b,[bermanfaat],positive


## Stemming

Untuk mengurangi variasi bentuk kata, dilakukan proses **stemming**, yaitu mengubah kata berimbuhan menjadi bentuk dasarnya. Proses ini bertujuan untuk mengurangi jumlah variasi kata yang merepresentasikan makna yang sama sehingga dapat menurunkan **dimensionalitas fitur** dan meningkatkan konsistensi representasi kata.

Untuk meningkatkan efisiensi proses stemming, digunakan **cache dictionary (`stem_cache`)** yang menyimpan hasil stemming kata yang telah diproses sebelumnya. Dengan demikian, jika kata yang sama muncul kembali, sistem dapat langsung mengambil hasil stemming dari cache tanpa perlu melakukan proses stemming ulang.

In [ ]:
stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

# Cache dictionary untuk menyimpan hasil stemming
# agar proses stemming lebih cepat (menghindari stemming ulang kata yang sama)
stem_cache = {}

In [ ]:
# Melakukan stemming pada setiap token menggunakan Sastrawi
# tqdm digunakan untuk menampilkan progress bar
df["content"] = df["content"].progress_apply(lambda tokens: [get_stemmed_term(t) for t in tokens])

df.sample(10, random_state=2)

100%|██████████| 13969/13969 [22:55<00:00, 10.15it/s]


,reviewId,content,sentiment
9026,1b0be493-10ef-4053-9708-bb449f4a902c,"[susah, masuk, frefikasih, wajah, kayal, taikkkk]",negative
11681,fb39dc4b-0e30-4600-b6a8-6dcbd33119a0,[eror],negative
10855,bd3f91b6-254d-4286-8138-5f08809f3445,"[woi, buka, ku, uninstal, pas, instal, ulang, ...",negative
770,74301678-3701-4d6c-aed4-6eca88a67c35,"[hilang, tibaknpadisaat, urgent, guna]",negative
3406,9019166f-5afd-45ed-a42b-c4019d286aaf,"[masyaa, allah, puas, prlayanannya]",positive
3262,701974be-4b7d-4758-9e09-3ecea8e31e41,"[simpel, mudah, digunakanbebas, biaya, pengiri...",positive
2562,b76d0167-2e6b-46a9-879e-72f025daf946,"[bagus, mudah, guna, mantap]",positive
5955,4a28e683-1d5b-4a1d-b9d7-4e324b48ef50,"[transaksi, jaring, stabil, gangguanleg, trans...",negative
10174,69b99d7e-6207-4ecb-b718-8e3aef084c1a,"[guna, kecewa, sidik, jari, print, kenal, pass...",negative
9604,4857ddf0-0726-4b0d-8d4f-3176a4ec1e1b,[manfaat],positive


# Save

## Save Data

Setelah seluruh tahap preprocessing selesai, dataset yang telah diproses disimpan ke dalam file CSV baru. Dataset ini kemudian digunakan sebagai input pada tahap **pelatihan model klasifikasi sentimen**.

In [ ]:
# Menggabungkan kembali token menjadi kalimat
df["content"] = df["content"].apply(lambda tokens: " ".join(tokens))

# Menyimpan hasil preprocessing ke file CSV baru
df.to_csv("clean_review_bsi_mobile.csv", index=False)

## Save Other Objects

Untuk memastikan konsistensi antara proses pelatihan model dan proses **inference**, daftar stopwords dan cache stemming yang digunakan selama preprocessing disimpan dalam bentuk file pickle. Artefak ini nantinya digunakan kembali pada tahap inference sehingga proses preprocessing pada data baru mengikuti konfigurasi yang sama dengan data pelatihan.

In [ ]:
# Menyimpan daftar stopword yang digunakan agar dapat direuse pada proses inference
with open("stopwords.pkl", "wb") as file:
  pickle.dump(listOfStopwords, file)

# Menyimpan cache stemming untuk mempercepat proses inference
with open("stem_cache.pkl", "wb") as file:
  pickle.dump(stem_cache, file)